<a href="https://colab.research.google.com/github/adharshkamath/syncode/blob/popl/syncode_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install syncode numexpr

In [ ]:
# common imports
from syncode import Syncode
from lark import Lark
import numexpr as ne
import ast

model_name = "microsoft/phi-2"

In [ ]:
# utility function common across tasks

def compute_answer(prompt, model):
    response = model.infer(prompt)[0]
    response = response.strip()
    try:
        result = ne.evaluate(response)
    except Exception as e:
        result = f"The evaluation of the code throws an exception."
    return response, result

# Task 0
### Use Syncode with the grammar below, to generate an arithemetic expression

In [ ]:
grammar = """
start: operand operator operand

operand: NUMBER
operator: ADD | SUB | MUL | DIV

ADD: "+"
SUB: "-"
MUL: "*"
DIV: "/"

%ignore " "
%import common.NUMBER
"""

In [ ]:
syn_llm = Syncode(model=model_name, grammar=grammar, max_new_tokens=20, mode='grammar_strict', do_sample=False)

In [ ]:
llm = Syncode(model=model_name, grammar=grammar, max_new_tokens=20, mode='original', do_sample=False)

In [ ]:
prompt = "Question: What is 221.5 multiplied by 10.23?\nAnswer: "
print("Prompt: ")
print(prompt)

In [ ]:
response, result = compute_answer(prompt, syn_llm)
print("LLM output with Syncode: ", response if len(response) > 0 else "<empty string>")
print("Final result with Syncode: ", result)

In [ ]:
response, result = compute_answer(prompt, llm)
print("LLM output without Syncode: ", response if len(response) > 0 else "<empty string>")
print("Final result without Syncode: ", result)

In [ ]:
del syn_llm, llm

# Task 1
### Extend the grammar to include operator precedence

In [ ]:
# Multiplication and Division have higher precedence
# than Addition and Subtraction.
# HINT: `expression` can involve Addition and Subtraction, while
# `term` involves Multiplication and Division

# grammar_precedence = """
# start: expression
# expression: ...
# ...
# ...
# term
# ...
# ...
# ADD: "+"
# SUB: "-"
# MUL: "*"
# DIV: "/"
# %ignore " "
# %import common.NUMBER
# """
grammar_precedence = ""

In [ ]:
syn_llm = Syncode(model=model_name, grammar=grammar_precedence, max_new_tokens=20, mode='grammar_strict', do_sample=False)

In [ ]:
llm = Syncode(model=model_name, grammar=grammar_precedence, max_new_tokens=20, mode='original', do_sample=False)

In [ ]:
instructions2 = """Given an English description of an arithmetic expression, output the corresponding mathematical expression.
Output only the mathematical expression, nothing else. Use parentheses to indicate the order of operations where necessary.
By default, multiplication and division get higher precedence, than addition and subtraction. Below are some examples, for reference:
Question: What is 45.1 plus 23.54, plus 10, minus 5?
Answer: 45.1 + 23.54 + 10 - 5
Question: What is 120.4 divided by 4.0, multiplied by 2?
Answer: 120.4 / 4.0 * 2
Question: What is 327. multiplied by 11.0, divided by 2?
Answer: 327.0 * 11.0 / 2
Question: What is 24 added to 35, all divided by 2?
Answer: (24 + 35) / 2
-----
"""

In [ ]:
prompt = instructions2 + "Convert the below statement to a mathematical expression in prefix notation.\nQuestion: What is 95 times 3, plus 10? \nAnswer: "

print("Prompt: ")
print(prompt)

print("-----")
response, result = compute_answer(prompt, syn_llm)
print("LLM output with Syncode: ", response if len(response) > 0 else "<empty string>")
print("Final result with Syncode: ", result)

print("-----")
response, result = compute_answer(prompt, llm)
print("LLM output without Syncode: ", response if len(response) > 0 else "<empty string>")
print("Final result without Syncode: ", result)

In [ ]:
del syn_llm, llm

# Bonus task
### Extend the grammar to include function calls

In [ ]:
# HINT: Add a terminal `function` which accepts all the unary functions we want in the grammar that is,
# function: "exp" | "log" | "sin" | "cos" | "tan" | "sqrt"

# grammar_fin = """
# start: expression

# expression: term
#         | expression ADD term
#         | expression SUB term

# term: factor
#     | term MUL factor
#     | term DIV factor

# factor: NUMBER
#       | "(" expression ")"
# ...
# ...

# ADD: "+"
# SUB: "-"
# MUL: "*"
# DIV: "/"

# %ignore " "
# %import common.NUMBER
# """
grammar_fin = ""

In [ ]:
syn_llm = Syncode(model=model_name, grammar=grammar_fin, max_new_tokens=20, mode='grammar_strict', do_sample=False)

In [ ]:
llm = Syncode(model=model_name, grammar=grammar_fin, max_new_tokens=20, mode='original', do_sample=False)

In [ ]:
instructions3 = """Given an English description of an arithmetic expression, output the corresponding mathematical expression.
Output only the mathematical expression, nothing else. Use parentheses to indicate the order of operations. Below are some examples, for reference:
Question: What is sine of 23 added to 21?
Answer: sin(23 + 21)
Question: What is exp of 2 added to 71, multiplied by sin of 23?
Answer: exp(2 + 71) * sin(23)
-----
"""

In [ ]:
prompt = instructions3 + "Question: What is cosine of 20, multiplied by 91, all multiplied by 2?\nAnswer: "

print("Prompt: ")
print(prompt)
print("-----")

response, result = compute_answer(prompt, syn_llm)
print("LLM output with Syncode: ", response if len(response) > 0 else "<empty string>")
print("Final result with Syncode: ", result)

print("-----")
response, result = compute_answer(prompt, llm)
print("LLM output without Syncode: ", response if len(response) > 0 else "<empty string>")
print("Final result without Syncode: ", result)